# Colon

In [ ]:
# ============================================================
# 5x5 CV with FIXED BEST HYPERPARAMETERS
# Reports:
#   - Accuracy mean ± std
#   - ROC-AUC mean ± std
#   - Macro-F1 mean ± std
#   - Total runtime
# Runs on cuda:7
# ============================================================

import os
# ---- MUST be set before torch import ----
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, gc, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# ---- IMPORTANT: import from your single file ----
from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

# -----------------------
# Config
# -----------------------
SEED = 42
DATA_FILE  = "coloncancer_encoded.csv"
TARGET_COL = "label"

# fixed best hyperparameters from Optuna
BEST_PARAMS = {
    "go_metric": "euclidean",
    "go_num_clusters": 10,
    "go_refine_passes": 3,
    "go_direction_select": True,
    "nsc_segmentation": "equal_mass",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 1.7570143129240916,
    "nsc_beta": 0.2244046472232107,
    "nsc_Mmin": 64,
    "nsc_Mmax": 384,
    "nsc_lmin": 16,
    "assume_standardized": False,
    "tabpfn_seed": 42,
}

# visible GPU is remapped to cuda:0 after CUDA_VISIBLE_DEVICES=7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Utils
# -----------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_multiclass_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map, int(len(class_map))

def compute_deltas_adjacent_corr(X_tr: np.ndarray, Pi_star: list[int], eps: float = 1e-12) -> torch.Tensor:
    """
    delta[t] = 1 - |corr(feature_{Pi[t]}, feature_{Pi[t+1]})|
    Returns torch.Tensor shape (m-1,) on CPU.
    """
    X = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)
    Xp = X[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std
    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()

def safe_multiclass_auc(y_true, proba, num_classes):
    """
    Binary: standard ROC-AUC on positive class
    Multiclass: macro OVR ROC-AUC
    """
    try:
        if num_classes == 2:
            if proba.ndim == 2 and proba.shape[1] == 2:
                return float(roc_auc_score(y_true, proba[:, 1]))
            else:
                return float("nan")
        else:
            y_bin = label_binarize(y_true, classes=np.arange(num_classes))
            return float(roc_auc_score(y_bin, proba, average="macro", multi_class="ovr"))
    except Exception:
        return float("nan")

# -----------------------
# Load data
# -----------------------
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)
y_raw = df[TARGET_COL].to_numpy()
X_df  = df.drop(columns=[TARGET_COL])

y_all, class_map, NUM_CLASSES = ensure_multiclass_contiguous(y_raw)

# same as your Optuna cell: global standardization first
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)

X_all = np.asarray(X_scaled, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] {DATA_FILE} | X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print(f"[CUDA] torch.cuda.is_available() = {torch.cuda.is_available()}")
print(f"[CUDA] torch.cuda.device_count()   = {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"[CUDA] current_device            = {torch.cuda.current_device()}")
    print(f"[CUDA] device_name               = {torch.cuda.get_device_name(0)}")
    print(f"[CUDA] using logical device      = {DEVICE}  (this is physical GPU 7 due to CUDA_VISIBLE_DEVICES=7)")

# 5x5 CV
rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

# -----------------------
# GO-LR once on FULL X
# -----------------------
seed_everything(SEED)
t0_total = time.time()

go = GraphFeatureOrdering(
    num_clusters=int(BEST_PARAMS["go_num_clusters"]),
    metric=BEST_PARAMS["go_metric"],
    refine=True,
    direction_select=bool(BEST_PARAMS["go_direction_select"]),
    refine_passes=int(BEST_PARAMS["go_refine_passes"]),
)

t0_go = time.time()
try:
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=False)
except RuntimeError:
    cleanup_cuda()
    Pi_star, _, _, _ = go.fit(X_all, seed=SEED, deterministic=True, use_cpu_kmeans=True)
t1_go = time.time()

print(f"[GO-LR] done in {t1_go - t0_go:.2f} sec | len(Pi_star)={len(Pi_star)}")

# -----------------------
# Fixed TabPFN config
# -----------------------
task_type = "binary" if NUM_CLASSES == 2 else "multiclass"

head_cfg = TabPFN25Config(
    task_type=task_type,
    num_classes=int(NUM_CLASSES),
    device=DEVICE,
    random_state=int(BEST_PARAMS["tabpfn_seed"]),
)

# -----------------------
# 5x5 CV evaluation
# -----------------------
accs, aucs, f1s, fold_times = [], [], [], []

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
    t0_fold = time.time()

    X_tr = X_all[tr_idx]
    y_tr = y_all[tr_idx]
    X_va = X_all[va_idx]
    y_va = y_all[va_idx]

    # NSC config on TRAIN only, with fixed Pi_star
    nsc = PIDFSegPCA(
        segmentation=BEST_PARAMS["nsc_segmentation"],
        l_min=int(BEST_PARAMS["nsc_lmin"]),
        m_rule=BEST_PARAMS["nsc_m_rule"],
        gamma=float(BEST_PARAMS["nsc_gamma"]),
        beta=float(BEST_PARAMS["nsc_beta"]),
        tau=float(BEST_PARAMS["nsc_tau"]),
        M_min=int(BEST_PARAMS["nsc_Mmin"]),
        M_max=int(BEST_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(BEST_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    deltas = compute_deltas_adjacent_corr(X_tr, Pi_star) \
        if BEST_PARAMS["nsc_segmentation"] != "uniform" else None

    X_tr_t = torch.from_numpy(X_tr)
    X_va_t = torch.from_numpy(X_va)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(BEST_PARAMS["nsc_tau"]),
        deltas=deltas
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(X_va_t, mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    P = head.predict_proba(Z_va)
    pred = np.argmax(P, axis=1)

    acc = float(accuracy_score(y_va, pred))
    f1  = float(f1_score(y_va, pred, average="macro"))
    auc = safe_multiclass_auc(y_va, P, NUM_CLASSES)

    accs.append(acc)
    f1s.append(f1)
    aucs.append(auc)

    t1_fold = time.time()
    fold_times.append(t1_fold - t0_fold)

    print(
        f"[Fold {fold_id:02d}/25] "
        f"ACC={acc:.6f} | AUC={auc:.6f} | F1={f1:.6f} | "
        f"time={t1_fold - t0_fold:.2f}s"
    )

    cleanup_cuda()

t1_total = time.time()

# -----------------------
# Final report
# -----------------------
acc_mean, acc_std = float(np.mean(accs)), float(np.std(accs, ddof=1))
auc_mean, auc_std = float(np.nanmean(aucs)), float(np.nanstd(aucs, ddof=1))
f1_mean,  f1_std  = float(np.mean(f1s)),  float(np.std(f1s, ddof=1))

total_runtime = t1_total - t0_total
mean_fold_time = float(np.mean(fold_times))

print("\n" + "=" * 60)
print("FINAL 5x5 CV RESULTS")
print("=" * 60)
print(f"Accuracy : {acc_mean:.6f} ± {acc_std:.6f}")
print(f"ROC-AUC  : {auc_mean:.6f} ± {auc_std:.6f}")
print(f"Macro-F1 : {f1_mean:.6f} ± {f1_std:.6f}")
print(f"Runtime  : {total_runtime:.2f} sec  ({total_runtime/60.0:.2f} min)")
print(f"Avg/fold : {mean_fold_time:.2f} sec")
print("=" * 60)

[DATA] coloncancer_encoded.csv | X=(62, 2000) | C=2 | map={0: 0, 1: 1}
[CUDA] torch.cuda.is_available() = True
[CUDA] torch.cuda.device_count()   = 1
[CUDA] current_device            = 0
[CUDA] device_name               = NVIDIA TITAN RTX
[CUDA] using logical device      = cuda  (this is physical GPU 7 due to CUDA_VISIBLE_DEVICES=7)
[GO-LR] done in 28.79 sec | len(Pi_star)=2000
[Fold 01/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=20.70s
[Fold 02/25] ACC=0.846154 | AUC=0.900000 | F1=0.837500 | time=16.89s
[Fold 03/25] ACC=0.916667 | AUC=1.000000 | F1=0.911111 | time=17.92s
[Fold 04/25] ACC=0.833333 | AUC=0.843750 | F1=0.812500 | time=13.93s
[Fold 05/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=21.85s
[Fold 06/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=22.07s
[Fold 07/25] ACC=0.615385 | AUC=0.675000 | F1=0.575163 | time=18.78s
[Fold 08/25] ACC=1.000000 | AUC=1.000000 | F1=1.000000 | time=28.81s
[Fold 09/25] ACC=0.833333 | AUC=0.937500 | F1=0.812500 | time=29.09